# HyDE (hypothetical doc embeddings)

> Notebook generated from [`examples/rag/04_hyde.py`](../../examples/rag/04_hyde.py).

| Data | Value |
|------|-------|
| **Dataset** | MS MARCO (embedded, Bing queries) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** Generates the hypothetical document before retrieval and compares recall vs direct query.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
HyDE — Hypothetical Document Embeddings for abstract search
================================================================
Architecture: SPEC-RAG-003 / prismal.rag.hyde

Dataset: MS MARCO (Microsoft MAchine Reading COmprehension)
  • 1 million real user questions from Bing with answer passages.
  • Reference: https://huggingface.co/datasets/microsoft/ms_marco
  • Why: MS MARCO contains many abstract questions like "why?"
    and "how?" where there is a semantic gap between the query and the documents.
    HyDE closes this gap by generating a hypothetical document that more
    closely resembles the real document than the original query.

HyDE architecture description:
  Instead of embedding the query directly:
  1. The LLM generates a hypothetical document that would answer the query.
  2. This hypothetical document is embedded (with better semantic coverage).
  3. ChromaDB is searched using the hypothetical document's embedding.

  Benefit: The hypothetical document uses the vocabulary and style
  of the corpus documents, not that of queries. This reduces
  the "vocabulary mismatch" in abstract searches.

Included comparison:
  - Standard RAG (embedding the query directly) vs HyDE
  - Especially effective for abstract/conceptual questions

Usage:
    uv run python examples/rag/04_hyde.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio

from langchain_core.documents import Document

from prismal.rag.crag import CRAGPipeline
from prismal.rag.hyde import HyDEResult, HyDERetriever
from prismal.rag.vector_store import ChromaVectorStore

## Dataset: MS MARCO passages on AI and technology

In [ ]:
MSMARCO_PASSAGES = [
    {
        "source": "msmarco_attention_mechanism",
        "passage": (
            "The attention mechanism in neural networks allows the model to focus on "
            "relevant parts of the input when producing each output token. Unlike "
            "traditional sequence-to-sequence models that compress all input information "
            "into a single fixed-size vector, attention maintains a dynamic context. "
            "The transformer attention computes similarity scores between all pairs of "
            "positions, enabling the model to directly attend to any position in the "
            "input sequence regardless of distance. This eliminates the information "
            "bottleneck that limited earlier RNN-based models."
        ),
    },
    {
        "source": "msmarco_gradient_descent",
        "passage": (
            "Gradient descent is the optimization algorithm used to train neural networks. "
            "It minimizes the loss function by iteratively adjusting model parameters in "
            "the direction of the negative gradient. Stochastic gradient descent (SGD) "
            "uses a random subset (mini-batch) of training examples for each update, "
            "making it computationally feasible for large datasets. Adam optimizer "
            "combines momentum (exponential moving average of gradients) with "
            "adaptive learning rates per parameter, typically outperforming vanilla SGD "
            "on deep learning tasks."
        ),
    },
    {
        "source": "msmarco_overfitting",
        "passage": (
            "Overfitting occurs when a model learns the training data too well, "
            "capturing noise and random fluctuations rather than the underlying pattern. "
            "An overfit model shows low training error but high test error. "
            "Regularization techniques prevent overfitting: L1 regularization adds "
            "absolute value of parameters to loss, promoting sparsity. L2 regularization "
            "adds squared parameters, shrinking weights toward zero. Dropout randomly "
            "deactivates neurons during training, acting as an ensemble method. "
            "Data augmentation artificially increases training set diversity."
        ),
    },
    {
        "source": "msmarco_transfer_learning",
        "passage": (
            "Transfer learning leverages knowledge gained from one task to improve "
            "performance on a related task. In NLP, large pre-trained language models "
            "like BERT and GPT are fine-tuned on domain-specific datasets, requiring "
            "significantly less labeled data. The pre-training phase captures general "
            "linguistic patterns, while fine-tuning specializes the model. This approach "
            "has democratized NLP by enabling small organizations to achieve state-of-the-art "
            "results without massive computational resources."
        ),
    },
    {
        "source": "msmarco_embedding_space",
        "passage": (
            "Word embeddings represent words as dense vectors in a continuous space "
            "where semantic similarity corresponds to geometric proximity. Word2Vec "
            "demonstrated that arithmetic operations on embeddings capture semantic "
            "relationships: king - man + woman = queen. Modern contextual embeddings "
            "from BERT produce different vectors for the same word in different contexts. "
            "Sentence transformers extend this to full sentence representations, "
            "enabling semantic search, duplicate detection, and textual entailment."
        ),
    },
    {
        "source": "msmarco_rag_benefits",
        "passage": (
            "Retrieval-Augmented Generation addresses the knowledge cutoff limitation "
            "of parametric language models by grounding responses in up-to-date documents. "
            "Instead of encoding all knowledge in model parameters, RAG retrieves relevant "
            "documents at inference time. This reduces hallucinations because the model "
            "explicitly uses factual sources. RAG also improves interpretability since "
            "retrieved chunks can be shown to users as citations. The tradeoff is "
            "increased latency from the retrieval step and dependency on retrieval quality."
        ),
    },
]

## MS MARCO queries — abstract vs. concrete

In [ ]:
MSMARCO_QUERIES = [
    # ABSTRACT queries — HyDE should help
    {
        "id": "MQ1",
        "query": "Why are transformers better than RNNs for NLP?",
        "type": "abstract",
        "expected_source": "msmarco_attention_mechanism",
        "description": "Explanatory question — HyDE generates a hypothetical document with 'because'",
    },
    {
        "id": "MQ2",
        "query": "How do modern neural networks prevent overfitting?",
        "type": "abstract",
        "expected_source": "msmarco_overfitting",
        "description": "Mechanism question — different vocabulary between query and document",
    },
    {
        "id": "MQ3",
        "query": "Why is transfer learning so powerful for NLP?",
        "type": "abstract",
        "expected_source": "msmarco_transfer_learning",
        "description": "Reasoning question — HyDE closes the semantic gap",
    },
    # CONCRETE queries — standard RAG should work well
    {
        "id": "MQ4",
        "query": "gradient descent optimizer Adam learning rate",
        "type": "concrete",
        "expected_source": "msmarco_gradient_descent",
        "description": "Technical query with exact keywords — standard RAG works well",
    },
    {
        "id": "MQ5",
        "query": "word embeddings semantic similarity word2vec",
        "type": "concrete",
        "expected_source": "msmarco_embedding_space",
        "description": "Technical keywords with direct lexical match",
    },
]


async def setup_hyde(collection_name: str = "msmarco_hyde") -> tuple[HyDERetriever, CRAGPipeline]:
    """Set up HyDE and standard CRAG for comparison.

    Returns:
        Tuple (HyDERetriever, CRAGPipeline) sharing the same vector store.
    """
    store = ChromaVectorStore(collection_name=collection_name)

    docs = [
        Document(
            page_content=p["passage"],
            metadata={
                "source": p["source"],
                "chunk_id": str(i),
                "dataset": "MS_MARCO",
            },
        )
        for i, p in enumerate(MSMARCO_PASSAGES)
    ]

    store.add_documents(docs)
    print(f"  ✓ {len(docs)} MS MARCO passages indexed")

    # Custom prompt to generate hypothetical documents in English
    hypothesis_prompt = (
        "You are an expert in artificial intelligence and machine learning. "
        "Write a concise technical paragraph (3-5 sentences) that directly "
        "answers the following question, as if it were an excerpt from "
        "an academic article or technical documentation: {query}"
    )

    hyde_retriever = HyDERetriever(
        vector_store=store,
        hypothesis_prompt=hypothesis_prompt,
    )

    crag_pipeline = CRAGPipeline(vector_store=store)

    return hyde_retriever, crag_pipeline


def print_comparison(query_info: dict, hyde_result: HyDEResult, crag_chunks: list) -> None:
    """Compare HyDE vs standard CRAG results."""
    print(f"\n[{query_info['id']}] Type: {query_info['type'].upper()}")
    print(f"  Query   : {query_info['query']}")
    print(f"  Purpose : {query_info['description']}")

    # Hypothetical document generated
    print("\n  📝 Hypothetical document generated by HyDE:")
    print(f"     {hyde_result.hypothesis[:200]}...")

    # Comparison of retrieved chunks
    hyde_sources = [c.source for c in hyde_result.chunks[:3]]
    crag_sources = [c.source for c in crag_chunks[:3]]

    print("\n  Retrieved chunks:")
    print(f"    HyDE (hypothetical): {hyde_sources}")
    print(f"    CRAG (direct query): {crag_sources}")

    # Verify expected source
    expected = query_info.get("expected_source")
    if expected:
        hyde_found = expected in hyde_sources
        crag_found = expected in crag_sources
        print(f"\n  Expected source ({expected}):")
        print(f"    HyDE found: {'✓' if hyde_found else '✗'}")
        print(f"    CRAG found: {'✓' if crag_found else '✗'}")

        # For abstract queries, HyDE should win
        if query_info["type"] == "abstract" and hyde_found and not crag_found:
            print("    → HyDE beats CRAG on abstract query ✓")
        elif query_info["type"] == "concrete" and crag_found:
            print("    → CRAG works well for concrete query ✓")

    print("\n  Top-2 HyDE chunks:")
    for chunk in hyde_result.chunks[:2]:
        print(f"    [{chunk.relevance_score:.2f}] {chunk.source}: {chunk.content[:80]}...")
    print("─" * 70)


async def main() -> None:
    print("=" * 70)
    print("  HyDE (Hypothetical Document Embeddings) — Dataset: MS MARCO")
    print("=" * 70)

    # Initialization
    print("\n[Initialization]")
    hyde_retriever, crag_pipeline = await setup_hyde()
    print("  ✓ HyDE Retriever and CRAG Pipeline initialized")

    # Explanation of the HyDE flow
    print("\n[HyDE vs standard RAG flow]")
    print("  Standard RAG:")
    print("    Query → embed(query) → similarity_search → chunks")
    print()
    print("  HyDE:")
    print("    Query → LLM generates hypothetical document H")
    print("         → embed(H) → similarity_search → chunks")
    print("  ")
    print("  The key: embed(H) ≈ embed(real_document) >> embed(query)")

    # Comparison
    print(f"\n[HyDE vs CRAG comparison on {len(MSMARCO_QUERIES)} MS MARCO queries]")

    hyde_wins = 0
    crag_wins = 0

    for query_info in MSMARCO_QUERIES:
        # Run HyDE
        hyde_result = await hyde_retriever.retrieve(query_info["query"], k=3)

        # Run CRAG for comparison
        crag_result = await crag_pipeline.run(query_info["query"])

        print_comparison(query_info, hyde_result, crag_result.sources)

        # Count wins (simplified)
        expected = query_info.get("expected_source")
        if expected:
            hyde_found = expected in [c.source for c in hyde_result.chunks[:3]]
            crag_found = expected in [c.source for c in crag_result.sources[:3]]

            if query_info["type"] == "abstract":
                if hyde_found:
                    hyde_wins += 1
                elif crag_found:
                    crag_wins += 1
            else:
                if crag_found:
                    crag_wins += 1
                elif hyde_found:
                    hyde_wins += 1

    # Summary
    print("\n[Comparison summary]")
    print(
        f"  Abstract queries (HyDE expected to win): {sum(1 for q in MSMARCO_QUERIES if q['type'] == 'abstract')}"
    )
    print(
        f"  Concrete queries (CRAG expected to win) : {sum(1 for q in MSMARCO_QUERIES if q['type'] == 'concrete')}"
    )

    print("\n[When to use HyDE?]")
    recommendations = [
        ("✓ Use HyDE  ", "Abstract questions: 'why?', 'how does it work?'"),
        ("✓ Use HyDE  ", "Query vocabulary very different from the corpus"),
        ("✓ Use HyDE  ", "Technical domain with specialized jargon"),
        ("✗ Avoid HyDE", "Queries with exact keywords from the corpus"),
        ("✗ Avoid HyDE", "Critical latency (HyDE adds 1 LLM call)"),
        ("✗ Avoid HyDE", "Limited token budget"),
    ]
    for status, case in recommendations:
        print(f"  {status}: {case}")

    # Show hypothetical embedding (first tokens)
    print("\n[Example of generated hypothetical document]")
    sample_result = await hyde_retriever.retrieve(
        "Why does self-attention outperform cross-attention in many cases?", k=2
    )
    print("  Query    : 'Why does self-attention outperform cross-attention?'")
    print(f"  Hypothesis: {sample_result.hypothesis[:250]}...")
    print(f"  Embedding dim: {len(sample_result.hypothesis_embedding)} hypothetical tokens")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()